In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from dotenv import load_dotenv
import os
import sys

load_dotenv()

import os

PROJECT_ROOT = os.getenv("PROJECT_ROOT")
sys.path.append(PROJECT_ROOT)

In [108]:
# load data filepaths from config.yaml

import yaml
import os

with open(os.path.join(os.getenv("PROJECT_ROOT"), 'config.yaml'), 'r') as file:
    config = yaml.safe_load(file)

# replace the ${PROJECT_ROOT} with the actual project root
for dataset_name in config['dataset_names']:
        for key, value in config['data'][dataset_name].items():
            config['data'][dataset_name][key] = value.replace('${PROJECT_ROOT}', os.getenv("PROJECT_ROOT"))

In [109]:
from src.load_data import load_data

def load_all_data(config):
    """
    Load all datasets specified in the config file.
    """
    data = dict()

    for dataset_name in config['dataset_names']:
        data[dataset_name] = dict()
        for split in ["train", "dev", "test"]:
            is_varierrnli = (dataset_name == "VariErrNLI")
            is_test = (split == "test")
            file_path = config['data'][dataset_name][f"{split}_file"]

            soft_labels, perspectivism, annotators_per_entry, ids, full_data = load_data(file_path, is_varierrnli=is_varierrnli, is_test=is_test)

            data[dataset_name][split] = {
                "soft_labels": soft_labels,
                "perspectivism": perspectivism,
                "annotators_per_entry": annotators_per_entry,
                "ids": ids,
                "data": full_data
            }

            print(f"Loaded {dataset_name} {split} data with {len(full_data)} examples.")
    
    return data

data_all_datasets = load_all_data(config)

Loaded CSC train data with 5628 examples.
Loaded CSC dev data with 704 examples.
Loaded CSC test data with 704 examples.
Loaded MP train data with 12017 examples.
Loaded MP dev data with 3005 examples.
Loaded MP test data with 3756 examples.
Loaded Paraphrase train data with 400 examples.
Loaded Paraphrase dev data with 50 examples.
Loaded Paraphrase test data with 50 examples.
Loaded VariErrNLI train data with 388 examples.
Loaded VariErrNLI dev data with 50 examples.
Loaded VariErrNLI test data with 50 examples.


In [115]:
for k, v in data_all_datasets["VariErrNLI"]["dev"].items():
    print(f"{k}: {v}")  # Print first two elements of each key for brevity

soft_labels: [[[1.0, 0.0], [0.5, 0.5], [0.25, 0.75]], [[0.25, 0.75], [0.75, 0.25], [1.0, 0.0]], [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0]], [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0]], [[1.0, 0.0], [0.75, 0.25], [0.25, 0.75]], [[0.0, 1.0], [1.0, 0.0], [1.0, 0.0]], [[1.0, 0.0], [0.75, 0.25], [0.25, 0.75]], [[1.0, 0.0], [0.75, 0.25], [0.25, 0.75]], [[0.0, 1.0], [1.0, 0.0], [1.0, 0.0]], [[0.75, 0.25], [1.0, 0.0], [0.25, 0.75]], [[1.0, 0.0], [0.667, 0.333], [0.0, 1.0]], [[0.0, 1.0], [1.0, 0.0], [1.0, 0.0]], [[1.0, 0.0], [0.5, 0.5], [0.5, 0.5]], [[0.75, 0.25], [0.0, 1.0], [0.75, 0.25]], [[1.0, 0.0], [0.75, 0.25], [0.25, 0.75]], [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0]], [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0]], [[1.0, 0.0], [0.667, 0.333], [0.33299999999999996, 0.667]], [[0.5, 0.5], [1.0, 0.0], [0.0, 1.0]], [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0]], [[0.75, 0.25], [1.0, 0.0], [0.25, 0.75]], [[0.5, 0.5], [1.0, 0.0], [0.25, 0.75]], [[0.5, 0.5], [1.0, 0.0], [0.5, 0.5]], [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0]], [[0.0, 1.0],

In [180]:
# load the prompt template

import json

def load_prompt_template(file_path):
    """
    Load the prompt template from the config file.
    """
    data = json.load(open(file_path, 'r'))

    for dataset_name in config['dataset_names']:
        general = data["content"]
        dataset_specific = data["datasets"][dataset_name]
        introduction = general["introduction"].replace("${TASK_NAME}", dataset_specific["task_name"])
        task_description = general["task_description"].replace("${INPUT_FORMAT}", dataset_specific["input_format"]).replace("${RESPONSE_FORMAT}", dataset_specific["response_format"]).replace("${LABEL_EXPLANATION}", dataset_specific["label_explanation"])


        data["datasets"][dataset_name]["prompt_template"] = introduction + " " + task_description + " " + general["style"] + "\n" + general["examples"] + "\n" + general["input"]

        print(f"Propmt for task {dataset_name} is like:\n {data['datasets'][dataset_name]['prompt_template']}")

    return data


prompts = load_prompt_template(os.path.join(os.getenv("PROJECT_ROOT"), 'prompts/prompt_template.json'))

Propmt for task CSC is like:
 You are an expert in guessing my response against a sarcasn detecton task. Your task is to analyze and predict my response to a pair of context and response between <<< and >>>, and label it with an integer from 0 to 5 where 0 means not sarcastic at all and 6 means completely sarcastic. You should reply with only the label without any additional text.
Here are some of my previous responses:
${EXAMPLES}
<<<
${INPUT}
>>>
Propmt for task MP is like:
 You are an expert in guessing my response against a irony detection task. Your task is to analyze and predict my response to a pair of post and reply between <<< and >>>, and label it with 0 or 1 where 0 means not ironic and 1 means ironic. You should reply with only the label without any additional text.
Here are some of my previous responses:
${EXAMPLES}
<<<
${INPUT}
>>>
Propmt for task Paraphrase is like:
 You are an expert in guessing my response against a paraphrase quality assessment task. Your task is to a

In [181]:
import random
import openai
import datetime
import json
from tqdm import tqdm

def example_random_selection(train_data, annotator_id, n_shots):
    """
    Select n_shots examples randomly from the train data for a specific annotator.
    """
    example_ids = [train_data["ids"][i] for i, example in enumerate(train_data["annotators_per_entry"]) if annotator_id in example]
    example_ids = random.sample(example_ids, min(n_shots, len(example_ids)))
    if not example_ids:
        raise ValueError(f"No examples found for annotator {annotator_id} in the training data.")
    prompt_examples = []
    for example_id in example_ids:
        example = train_data["data"][example_id]
        label = example["annotations"][annotator_id]
        text =  "\n".join([f"[{k}]: {v}" for k, v in example["text"].items()])
        response = f"[label]: {label}"
        prompt_examples.append(f"{text}\n{response}\n")


    return example_ids, "\n".join(prompt_examples)


def icl_predict(dataset_name, test_mode, train_data, test_data, prompt, model, n_shots, selection_method):
    """
    intput:
        - dataset_name: name of the dataset
        - test_mode: "dev" or "test"
        - train_data: training data for the dataset
        - test_data: dev/test data for the dataset
        - prompt: prompt templates of all the datasets
        - model: model to use for prediction
        - n_shots: number of shots to use for in-context learning
        - selection_method: method to select examples for in-context learning
    output:
        - predictions: predictions for the test data
    """

    openai_api_key = os.environ.get("OPENAI_API_KEY")
    client = openai.OpenAI(api_key=openai_api_key)

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    logs = {
        "datetime": timestamp,
        "dataset_name": dataset_name,
        "test_mode": test_mode,
        "model": model,
        "n_shots": n_shots,
        "selection_method": selection_method,
        "predictions": dict(),
        "prompt_template": prompt["datasets"][dataset_name]["prompt_template"],
        "prompts": dict(),
        "examples_ids": dict()
    }

    predictions_pe = dict()

    for test_id in tqdm(test_data["ids"]):
        annotators = test_data["data"][test_id]["annotators"].split(",")
        predictions = list()
        for annotator in annotators:
            if selection_method == "random":
                example_ids, prompt_example = example_random_selection(train_data, annotator, n_shots)
            prompt_template = prompt["datasets"][dataset_name]["prompt_template"]
            prompt_template = prompt_template.replace("${EXAMPLES}", prompt_example)

            logs["examples_ids"][f"{test_id}+{annotator}"] = example_ids

            input_text = "\n".join([f"[{k}]: {v}" for k, v in test_data["data"][test_id]["text"].items()])
            input_text += ("\n" + "[label]:")

            prompt_template = prompt_template.replace("${INPUT}", input_text)

            logs["prompts"][f"{test_id}+{annotator}"] = prompt_template

            response = client.chat.completions.create(
                model = model,
                messages = [{
                    "role": "user",
                    "content": prompt_template
                }],
                temperature = 0
            ).choices[0].message.content
            response = response.replace("\n", "").replace("[label]:", "").strip()
            if dataset_name != "VariErrNLI":
                try:
                    response = int(response)
                except ValueError:
                    print(f"Warning: Response '{response}' is not an integer for test_id {test_id} and annotator {annotator}.")

            predictions.append(response)

        predictions_pe[test_id] = predictions

    logs["predictions"] = predictions_pe
    try:
        json.dump(logs, open(os.path.join(os.getenv("PROJECT_ROOT"), f"logs/icl_{dataset_name}_{test_mode}_{model}_{n_shots}_{selection_method}_{timestamp}.json"), "w"), indent=4)

        json.dump(predictions_pe, open(os.path.join(os.getenv("PROJECT_ROOT"), f"predictions/icl_{dataset_name}_{test_mode}_{model}_{n_shots}_{selection_method}_{timestamp}.json"), "w"), indent=4)
    except Exception as e:
        print(f"Error saving logs or predictions: {e}")
    return predictions_pe, logs

In [173]:
# demo test on all datasets excep for MP

model = "gpt-4o-mini"
n_shots = 10
selection_method = "random"
test_mode = "dev"

for dataset_name in config["dataset_names"]:
    if dataset_name != "Paraphrase":
        continue
    print(f"Running ICL for dataset {dataset_name}...")
    predictions_pe, logs = icl_predict(dataset_name, test_mode, data_all_datasets[dataset_name]["train"], data_all_datasets[dataset_name][test_mode], prompts, model, n_shots, selection_method)

Running ICL for dataset Paraphrase...


100%|██████████| 50/50 [02:15<00:00,  2.71s/it]


In [196]:
import json

def varierrnli_to_soft_labels_and_pe(predictions):
    soft_labels = dict()
    pe = dict()

    labels = ["contradiction", "entailment", "neutral"]

    for id, annotations in predictions.items():
        num_annotators = len(annotations)
        soft_label = {label: dict() for label in labels}
        for label in labels:
            count = sum(1
                        for ann in annotations
                        if label in [a.strip() for a in ann.split(',')])
            p1 = count / num_annotators
            p0 = 1 - p1
            soft_label[label] = {"0": p0, "1": p1}

        soft_labels[id] = soft_label

        
        # Initialize label vectors for each annotator
        label_vectors = {label: [0] * num_annotators for label in labels}

        # Fill in the label vectors
        for i, annotation_str in enumerate(annotations):
            for label in annotation_str.split(','):
                label = label.strip()
                if label in label_vectors:
                    label_vectors[label][i] = 1

        pe[id] = list(label_vectors.values())

    return soft_labels, pe

predictions_varierrnli = json.load(open(os.path.join(os.getenv("PROJECT_ROOT"), f"predictions/icl_VariErrNLI_dev_gpt-4o-mini_10_random_20250701_153208.json"), "r"))
predictions_soft_labels_varierrnli, predictions_pe_varierrnli = varierrnli_to_soft_labels_and_pe(predictions_varierrnli)
print(predictions_soft_labels_varierrnli["59208"])
print(predictions_pe_varierrnli)

{'contradiction': {'0': 0.75, '1': 0.25}, 'entailment': {'0': 1.0, '1': 0.0}, 'neutral': {'0': 0.25, '1': 0.75}}
{'49807': [[1, 1, 1, 1, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], '76020': [[0, 0, 0, 0], [1, 1, 1, 1], [0, 0, 0, 0]], '69815': [[0, 0, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1]], '94674': [[1, 1, 1, 1], [0, 0, 0, 0], [0, 0, 0, 0]], '113193': [[0, 0, 0, 0], [1, 1, 1, 1], [0, 0, 0, 0]], '65879': [[1, 1, 1, 1], [0, 0, 0, 0], [0, 0, 0, 0]], '107468': [[0, 0, 0, 0], [0, 0, 1, 0], [1, 1, 0, 1]], '119768': [[1, 1, 1, 0], [0, 0, 0, 0], [0, 0, 0, 1]], '105769': [[1, 1, 1, 1], [0, 0, 0, 0], [0, 0, 0, 0]], '52542': [[0, 0, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1]], '76947': [[0, 0, 0, 0], [1, 1, 1, 1], [0, 0, 0, 0]], '120149': [[0, 0, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1]], '59208': [[0, 0, 1, 0], [0, 0, 0, 0], [1, 1, 0, 1]], '80630': [[0, 1, 1, 1, 0, 1], [1, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0]], '46003': [[0, 0, 0, 0], [1, 1, 1, 1], [0, 0, 0, 0]], '58357': [[1, 1], [0, 0], [0, 0]], '91601': [[0, 0, 0], [0,

In [179]:
def pe_to_soft_labels(dataset_name, predictions_pe):
    """
    Convert predictions to soft labels.
    """
    results = dict()

    if dataset_name == "CSC":
        label_range = list(range(0, 6))
    elif dataset_name == "MP":
        label_range = [0, 1]
    else:
        label_range = list(range(-5, 6))
    count = {k: 0 for k in label_range}
    for k, v in predictions_pe.items():
        for label in v:
            count[label] += 1
        total = sum(count.values())
        soft_labels = [count[k] / total for k in label_range]
        results[k] = soft_labels

    return results

predictions_pe = json.load(open(os.path.join(os.getenv("PROJECT_ROOT"), f"predictions/icl_Paraphrase_dev_gpt-4o-mini_20_random_20250701_174650.json"), "r"))

predictions_soft_labels = pe_to_soft_labels("Paraphrase", predictions_pe)
predictions_soft_labels

{'108774': [0.0, 0.0, 0.5, 0.25, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 '211976': [0.375, 0.0, 0.375, 0.125, 0.125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 '394553': [0.25,
  0.08333333333333333,
  0.4166666666666667,
  0.16666666666666666,
  0.08333333333333333,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 '178525': [0.4375,
  0.0625,
  0.3125,
  0.125,
  0.0625,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0,
  0.0],
 '95504': [0.35, 0.05, 0.25, 0.1, 0.05, 0.0, 0.0, 0.0, 0.05, 0.0, 0.15],
 '330845': [0.3333333333333333,
  0.041666666666666664,
  0.2916666666666667,
  0.08333333333333333,
  0.041666666666666664,
  0.0,
  0.041666666666666664,
  0.0,
  0.041666666666666664,
  0.0,
  0.125],
 '292473': [0.2857142857142857,
  0.03571428571428571,
  0.32142857142857145,
  0.10714285714285714,
  0.03571428571428571,
  0.0,
  0.07142857142857142,
  0.0,
  0.03571428571428571,
  0.0,
  0.10714285714285714],
 '199987': [0.25,
  0.03125,
  0.28125,
  0.15625,
  0.03125,
  0.0,
  0.09375,
  0.0,
  0.0625,
  0.0,
  0.

In [225]:
from src.evaluation import soft_label_evaluation, perspectivist_evaluation

def evaluate(dataset_name, test_mode, full_data, predictions_soft_labels, predictions_pe):
    """
    Evaluate the predictions for a specific dataset and a specific list of entries.
    """

    ids = list(predictions_pe.keys())
    

    target_soft_labels = full_data[dataset_name][test_mode]["soft_labels"]
    target_pe = full_data[dataset_name][test_mode]["perspectivism"]

    if dataset_name != "VariErrNLI":
        soft_label_evaluation_results = soft_label_evaluation(dataset_name, target_soft_labels, list(predictions_soft_labels.values()))
        perspectivist_evaluation_results = perspectivist_evaluation(dataset_name, target_pe, list(predictions_pe.values()))
    else:
        soft_label_evaluation_results = soft_label_evaluation(dataset_name, target_soft_labels, [
           [
               list(q.values()) for _, q in v.items()
           ] for _, v in predictions_soft_labels.items()
        ])
        perspectivist_evaluation_results = perspectivist_evaluation(dataset_name, target_pe, [v for _, v in predictions_pe.items()])

    return {
        "soft_label_evaluation": soft_label_evaluation_results,
        "perspectivist_evaluation": perspectivist_evaluation_results
    }
    

In [200]:
dataset_name = "Paraphrase"
test_mode = "dev"

results = evaluate(dataset_name, test_mode, data_all_datasets, predictions_soft_labels, predictions_pe)
results

[[0.0, 0.25, 0.25, 0.0, 0.25, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.75, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.25], [0.25, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0], [0.5, 0.25, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.75, 0.0, 0.25], [0.0, 0.25, 0.25, 0.25, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0], [0.0, 0.25, 0.25, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.25], [0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.25, 0.25, 0.25], [0.75, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.5, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.25], [0.25, 0.25, 0.25, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.25, 0.0, 0.25, 0.0, 0.0, 0.0, 0.25, 0.0, 0.25, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.25, 0.5, 0.25], [0.75, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0], [0.75, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0], [0.25, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.25, 0.0], [0.0, 0.0, 0.75, 0.0, 0.0, 0.0, 0.0, 0.0, 0

{'soft_label_evaluation': 2.83587,
 'perspectivist_evaluation': 0.18545454545454543}

In [226]:
results_varierrnli = evaluate("VariErrNLI", "dev", data_all_datasets, predictions_soft_labels_varierrnli, predictions_pe_varierrnli)
results_varierrnli

[[[0, 0, 0, 0], [0, 1, 0, 1], [1, 1, 1, 0]], [[1, 0, 1, 1], [0, 1, 0, 0], [0, 0, 0, 0]], [[0, 0, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1]], [[0, 0, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1]], [[0, 0, 0, 0], [1, 0, 0, 0], [0, 1, 1, 1]], [[1, 1, 1, 1], [0, 0, 0, 0], [0, 0, 0, 0]], [[0, 0, 0, 0], [0, 1, 0, 0], [1, 0, 1, 1]], [[0, 0, 0, 0], [0, 0, 0, 1], [1, 1, 1, 0]], [[1, 1, 1, 1], [0, 0, 0, 0], [0, 0, 0, 0]], [[1, 0, 0, 0], [0, 0, 0, 0], [0, 1, 1, 1]], [[0, 0, 0], [0, 1, 0], [1, 1, 1]], [[1, 1, 1, 1], [0, 0, 0, 0], [0, 0, 0, 0]], [[0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 0, 1]], [[0, 0, 0, 1], [1, 1, 1, 1], [0, 0, 1, 0]], [[0, 0, 0, 0], [0, 0, 0, 1], [1, 1, 1, 0]], [[0, 0], [0, 0], [1, 1]], [[0, 0, 0], [0, 0, 0], [1, 1, 1]], [[0, 0, 0], [0, 1, 0], [1, 0, 1]], [[0, 1], [0, 0], [1, 1]], [[0, 0], [0, 0], [1, 1]], [[1, 0, 0, 0], [0, 0, 0, 0], [0, 1, 1, 1]], [[0, 0, 1, 1], [0, 0, 0, 0], [1, 1, 1, 0]], [[1, 1, 0, 0], [0, 0, 0, 0], [0, 0, 1, 1]], [[0, 0, 0, 0], [0, 0, 0, 0], [1, 1, 1, 1]], [[1, 1], [0, 0], [0, 0]]

TypeError: unsupported operand type(s) for -: 'int' and 'list'